In [ ]:
import pandas as pd
import os
# Check the working directory
os.getcwd()

In [ ]:
# reset the working directory
os.chdir('C:\\Users\\wanye\\Documents\\vacination_2026')

In [ ]:
# --- Setup ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)
torch.set_float32_matmul_precision("high")  # minor speedup on recent PyTorch

In [ ]:
import pandas as pd
# Read data into papers
# By using panda, a .csv file is read into a dataframe.
records = pd.read_csv('xxxx.csv')
records.head()

In [ ]:
    """
    Convert a period-delimited theme specification into
    a matrix of semantic head embeddings.

    Parameters
    ----------
    text : str
        Theme description, subthemes, and linguistic markers
        combined into a single string, with individual semantic
        units separated by periods.

    model : SentenceTransformer
        MPNet sentence-transformer model.

    Returns
    -------
    head_matrix : np.ndarray
        Shape (n_heads, 768). Each row is one normalized
        semantic head embedding.

    heads : list[str]
        Text corresponding to each row of head_matrix.
    """
import numpy as np


def head_encoding(text, model):

    # -----------------------------------------
    # 1. Split the specification into heads
    # -----------------------------------------
    heads = [
        chunk.strip()
        for chunk in text.split(".")
        if chunk.strip()
    ]

    if not heads:
        raise ValueError("No valid heads were found.")

    # -----------------------------------------
    # 2. Encode all heads as one batch
    # -----------------------------------------
    head_matrix = model.encode(
        heads,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    # Ensure float32 for efficient matrix operations
    head_matrix = head_matrix.astype(np.float32)

    return head_matrix, heads

In [ ]:
def state_projection(text, head_matrix, model):
    """
    Encode one text and compute its cosine similarity
    with every head.

    Parameters
    ----------
    text : str
        Input text to compare against the heads.

    head_matrix : np.ndarray
        Head embeddings from head_encoding().
        Shape: (n_heads, 768)

    model : SentenceTransformer
        Loaded sentence-transformer model.

    Returns
    -------
    similarity_vector : np.ndarray
        Cosine similarity with each head.
        Shape: (n_heads, 1)
    """

    if not isinstance(text, str) or not text.strip():
        raise ValueError("Input text must be a non-empty string.")

    # Encode input text
    post_vector = model.encode(
        [text],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    ).astype(np.float32)

    # post_vector.shape = (1, 768)

    # Compare against all heads
    similarity_vector = head_matrix @ post_vector.T

    # similarity_vector.shape = (n_heads, 1)

    return similarity_vector

In [ ]:
def activation(
    similarity_vector,
    threshold=0.4,
    midpoint=0.6,
    steepness=15
):
    """
    Convert an n x 1 similarity vector into a single
    Semantic Evidence Score (SES).

    Sequence
    --------
    1. Similarity <= threshold -> 0
    2. Apply shifted sigmoid to surviving similarities
    3. Aggregate activated evidence using the L2 norm

    Parameters
    ----------
    similarity_vector : np.ndarray
        Raw cosine similarities.
        Shape: (n, 1)

    threshold : float
        Similarities <= this value are discarded.

    midpoint : float
        Midpoint of the sigmoid transformation.

    steepness : float
        Controls the steepness of the sigmoid.

    Returns
    -------
    score : float
        L2 norm of the activated evidence.
    """

    scores = np.asarray(
        similarity_vector,
        dtype=np.float32
    ).copy()

    # 1. Threshold
    mask = scores > threshold

    # Start with all evidence set to zero
    activated = np.zeros_like(scores)

    # 2. Sigmoid only surviving similarities
    activated[mask] = 1.0 / (
        1.0 + np.exp(
            -steepness * (scores[mask] - midpoint)
        )
    )

    # 3. L2 aggregation
    score = float(np.linalg.norm(activated, ord=2))

    return score

In [ ]:
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

# ============================================================
# 1. SETUP
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

torch.set_float32_matmul_precision("high")

model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device=device
)

# ============================================================
# EXECUTION PIPELINE
# ============================================================

# Load theme descriptions
themes = pd.read_csv("themes.csv")


# ------------------------------------------------------------
# Iterate through themes
# ------------------------------------------------------------

for theme_idx, theme_row in themes.iterrows():

    theme_id = str(theme_row["theme_id"])
    theme_name = str(theme_row["theme_name"])
    theme_text = theme_row["theme_description"]


    # Skip missing/empty themes
    if pd.isna(theme_text) or not str(theme_text).strip():
        continue

    print(
        f"Processing theme {theme_idx + 1}/{len(themes)}: "
        f"{theme_id}"
    )

# ============================================================
# EXECUTION PIPELINE
# ============================================================

for theme_idx, theme_row in themes.iterrows():

    theme_id = str(theme_row["theme_id"])
    theme_text = theme_row["theme_text"]

    if pd.isna(theme_text) or not str(theme_text).strip():
        continue

    print(
        f"Processing theme {theme_idx + 1}/{len(themes)}: "
        f"{theme_id}"
    )

    # --------------------------------------------------------
    # HEAD ENCODING
    # --------------------------------------------------------

    head_matrix, heads = head_encoding(
        str(theme_text),
        model
    )

    n_heads = len(heads)

    # --------------------------------------------------------
    # OUTPUT COLUMNS
    # --------------------------------------------------------

    score_column = f"{theme_id}_score"
    norm_score_column = f"{theme_id}_norm_score"
    breadth_column = f"{theme_id}_evidence"

    records[score_column] = np.nan
    records[norm_score_column] = np.nan
    records[breadth_column] = 0


    # --------------------------------------------------------
    # ITERATE THROUGH POSTS
    # --------------------------------------------------------

    for post_idx, post_row in records.iterrows():

        text = post_row["content"]

        if pd.isna(text) or not str(text).strip():
            continue


        # n × 1 raw semantic similarities
        state_vector = state_projection(
            str(text),
            head_matrix,
            model
        )


        # Does this post contain qualifying evidence?
        has_evidence = np.any(
            state_vector > 0.40
        )


        # Threshold -> sigmoid -> L2
        evidence_score = activation(
            state_vector
        )


        # Normalize for number of heads
        normalized_score = (
            evidence_score / np.sqrt(n_heads)
        )


        # Store
        records.at[
            post_idx,
            score_column
        ] = evidence_score

        records.at[
            post_idx,
            norm_score_column
        ] = normalized_score

        records.at[
            post_idx,
            breadth_column
        ] = int(has_evidence)


print("Processing complete.")

In [ ]:
# ============================================================
# BUILD BREADTH × INTENSITY THEME PROFILE
# ============================================================

theme_profiles = []


for _, theme_row in themes.iterrows():

    theme_id = str(theme_row["theme_id"])

    evidence_column = f"{theme_id}_evidence"
    norm_score_column = f"{theme_id}_norm_score"

    if evidence_column not in records.columns:
        continue


    # --------------------------------------------------------
    # BREADTH
    #
    # Proportion of valid posts containing at least
    # one qualifying semantic head
    # --------------------------------------------------------

    valid_mask = records["content"].notna() & (
        records["content"].astype(str).str.strip() != ""
    )

    breadth = records.loc[
        valid_mask,
        evidence_column
    ].mean()


    # --------------------------------------------------------
    # INTENSITY
    #
    # Mean normalized semantic evidence among posts
    # where the theme is present
    # --------------------------------------------------------

    evidence_mask = (
        valid_mask &
        (records[evidence_column] == 1)
    )

    intensity = records.loc[
        evidence_mask,
        norm_score_column
    ].mean()


    # --------------------------------------------------------
    # SAVE PROFILE
    # --------------------------------------------------------

    theme_profiles.append({
        "theme_id": theme_id,
        "breadth": breadth,
        "intensity": intensity,
        "n_posts_with_evidence": int(
            evidence_mask.sum()
        ),
        "n_valid_posts": int(
            valid_mask.sum()
        )
    })


theme_profiles = pd.DataFrame(theme_profiles)

theme_profiles

In [ ]:
import matplotlib.pyplot as plt


# ============================================================
# ADD THEME NAMES TO THE PROFILE DATA
# ============================================================

theme_profiles = theme_profiles.merge(
    themes[["theme_id", "theme_name"]],
    on="theme_id",
    how="left"
)


# ============================================================
# BREADTH × INTENSITY PLOT
# ============================================================

plt.figure(figsize=(12, 8))

plt.scatter(
    theme_profiles["breadth"],
    theme_profiles["intensity"],
    s=80
)


# Label each point using the human-readable theme name
for _, row in theme_profiles.iterrows():

    plt.annotate(
        row["theme_name"],
        (
            row["breadth"],
            row["intensity"]
        ),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=9
    )


# ============================================================
# LABELS
# ============================================================

plt.xlabel(
    "Breadth (Proportion of Posts)"
)

plt.ylabel(
    "Intensity (Mean Normalized Semantic Evidence)"
)

plt.title(
    "Theme Prevalence: Breadth × Intensity"
)


# ============================================================
# AXIS RANGE
# ============================================================

plt.xlim(0, 1)
plt.ylim(0, 1)


# ============================================================
# DISPLAY
# ============================================================

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PREPARE TIME VARIABLE
# ============================================================

# Convert timestamp to datetime
records["date"] = pd.to_datetime(
    records["date"],
    errors="coerce"
)

# Define time period — monthly
records["period"] = records["date"].dt.to_period("M")


# ============================================================
# BUILD TIME-BASED THEME PROFILES
# ============================================================

time_profiles = []

for _, theme_row in themes.iterrows():

    theme_id = str(theme_row["theme_id"])
    theme_name = str(theme_row["theme_name"])

    evidence_col = f"{theme_id}_evidence"
    score_col = f"{theme_id}_norm_score"

    # Skip theme if required columns are missing
    if (
        evidence_col not in records.columns
        or score_col not in records.columns
    ):
        continue

    # --------------------------------------------------------
    # Calculate profile within each time period
    # --------------------------------------------------------

    for period, group in records.groupby(
        "period",
        observed=True
    ):

        # Skip invalid dates
        if pd.isna(period):
            continue

        # Breadth:
        # proportion of posts containing qualifying evidence
        breadth = group[evidence_col].mean()

        # Posts containing qualifying evidence
        evidence_posts = group[
            group[evidence_col] == 1
        ]

        # Intensity:
        # mean normalized semantic evidence among evidence posts
        intensity = evidence_posts[
            score_col
        ].mean()

        # Store time-period profile
        time_profiles.append({
            "theme_id": theme_id,
            "theme_name": theme_name,
            "period": str(period),
            "breadth": breadth,
            "intensity": intensity,
            "n_posts": len(group),
            "n_evidence_posts": len(evidence_posts)
        })


# ============================================================
# CREATE DATAFRAME
# ============================================================

time_profiles = pd.DataFrame(time_profiles)

In [ ]:
# ============================================================
# SELECT THEME
# ============================================================

theme_to_plot = "T01"

plot_data = time_profiles[
    time_profiles["theme_id"] == theme_to_plot
].copy()

if plot_data.empty:
    raise ValueError(
        f"No time-profile data found for theme_id: {theme_to_plot}"
    )

# Get human-readable theme name
theme_name = plot_data["theme_name"].iloc[0]


# ============================================================
# SORT CHRONOLOGICALLY
# ============================================================

plot_data["period_dt"] = pd.to_datetime(
    plot_data["period"]
)

plot_data = plot_data.sort_values(
    "period_dt"
)


# ============================================================
# PLOT BREADTH × INTENSITY TRAJECTORY
# ============================================================

plt.figure(figsize=(10, 8))

plt.plot(
    plot_data["breadth"],
    plot_data["intensity"],
    marker="o"
)


# Label each point with its time period
for _, row in plot_data.iterrows():

    plt.annotate(
        row["period"],
        (
            row["breadth"],
            row["intensity"]
        ),
        xytext=(6, 6),
        textcoords="offset points",
        fontsize=9
    )


# ============================================================
# LABELS
# ============================================================

plt.xlabel(
    "Breadth (Proportion of Posts)"
)

plt.ylabel(
    "Intensity (Mean Normalized Semantic Evidence)"
)

plt.title(
    f"Theme Trajectory: {theme_name}"
)


# ============================================================
# AXIS RANGE
# ============================================================

plt.xlim(0, 1)
plt.ylim(0, 1)


# ============================================================
# DISPLAY
# ============================================================

plt.tight_layout()
plt.show()